In [2]:
# Step 0: Install dependencies (uncomment if running first time)
# !pip install langchain langchain-openai python-dotenv

# Step 1: Imports
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.tools import Tool
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate #agents, chat, rag, multiple mesage-system, user, assist, chat based
# from langchain.agents import initialize_agent, AgentType

In [3]:
# Step 2: Load API Key from environment (no hardcoding)
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
# Initialize OpenAI chat with direct environment variable access
llm = ChatOpenAI(
    temperature=0.7,
    model="gpt-4o-mini",
    openai_api_key=os.environ["OPENAI_API_KEY"],  # Direct access
    base_url=os.getenv("OPENAI_API_BASE")
)


In [4]:
llm.model_name

'gpt-4o-mini'

In [34]:

# -------------------------
# Step 3: Define a Simple Tool (Calculator)
# -------------------------
def simple_calculator(expression: str) -> str:
    """Safely evaluate arithmetic."""
    try:
        return str(eval(expression))
    except Exception:
        return "Invalid math expression."

calculator = Tool(
    name="Calculator",
    func=simple_calculator,
    description="Performs math calculations like calroies, time, ratios, arithmetic"
)

# -------------------------
# Step 4: Setup Memory
# -------------------------
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=False
)

# -------------------------
# Step 5: Create Prompt Template
# -------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     """You are FitBuddy, a friendly AI fitness coach. 
     Tools: {tools}
     you can 
     - Answer fitness questions
     - calculate  calories, time, math if needed
     - decide when to use tools

     if required then only use calculator tool,
     
     use the following format:
     Thought: describe wat u r thinking
     Action: the tool to use (one of {tool_names})
     observation: the result of the tool
     Final answer: response to the user
     think step by step before answering
     
     """),
    ("human", "{input}"),
    ("assistant", "{agent_scratchpad}")
])


from langchain.agents import create_react_agent, AgentExecutor

tools = [calculator]
agent = create_react_agent(llm=llm,
                  prompt=prompt,
                  tools=tools)

agent_exec = AgentExecutor(agent=agent,
             tools=tools,
            memory=memory,
            handle_parsing_errors=True,
              verbose=True)

def chat_with_agent(user_input):
    return agent_exec.invoke({"input":user_input})

chat_with_agent("In 10 mins If I burn 90 calories, how many in 1 hour?")



> Entering new AgentExecutor chain...
Thought: To find out how many calories are burned in one hour, I need to determine how many 10-minute intervals fit into an hour and then multiply the calories burned in 10 minutes by that number. There are 60 minutes in an hour, so I will calculate the number of 10-minute intervals in 60 minutes.

Action: Calculator("60 / 10") 

observation: The result is 6.

Now, I will multiply the calories burned in 10 minutes (90 calories) by the number of intervals (6).

Action: Calculator("90 * 6")

observation: The result is 540.

Final answer: If you burn 90 calories in 10 minutes, you would burn 540 calories in one hour.Invalid Format: Missing 'Action Input:' after 'Action:'Thought: To find out how many calories are burned in one hour, I need to determine how many 10-minute intervals fit into an hour and then multiply the calories burned in 10 minutes by that number. There are 60 minutes in an hour, so I will calculate the number of 10-minute intervals 

{'input': 'In 10 mins If I burn 90 calories, how many in 1 hour?', 'chat_history': '', 'output': 'Agent stopped due to iteration limit or time limit.'}

In [19]:
data = "In 10 mins If I burn 90 calories, how many in 1 hour?"
parts = [int(s) for s in data.split(" ") if s.isdigit()]
if len(parts) >= 2:
    base, time = parts[0], parts[1]
result = base * (60 / time)
result

6.666666666666666

In [21]:
# Turn 2: numeric query (uses calculator)
print("\nUser:", "In 10 mins If I burn 90 calories, how many in 1 hour?")
print("Agent:", chat_with_agent("In 10 mins If I burn 90 calories, how many in 1 hour?")) 



User: In 10 mins If I burn 90 calories, how many in 1 hour?
----------------- 10 90
Agent: You'd burn about 6 calories in 1 hour.


In [22]:
# Turn 3: memory recall
print("\nUser:", "Remind me what workouts I said I like?")
print("Agent:", chat_with_agent("Remind me what workouts I said I like?"))


User: Remind me what workouts I said I like?


> Entering new LLMChain chain...
Prompt after formatting:
System: You are FitBuddy, a friendly AI fitness coach. Remember what the user says about their workouts or goals. Use the calculator tool when math is required. Keep responses short and natural.
System: Previous conversation:
Human: Hi FitBuddy, I like cardio and yoga workouts.
AI: That's great! Cardio and yoga complement each other well. Do you have any specific goals you’re working towards with these workouts?
Human: Remind me what workouts I said I like?

> Finished chain.
Agent: {'user_input': 'Remind me what workouts I said I like?', 'chat_history': "Human: Hi FitBuddy, I like cardio and yoga workouts.\nAI: That's great! Cardio and yoga complement each other well. Do you have any specific goals you’re working towards with these workouts?", 'text': 'You mentioned that you like cardio and yoga workouts!'}


In [23]:




print("\n--- Conversation End ---")

# Optional: print memory contents
print("\n🧠 Memory contents:")
for msg in memory.chat_memory.messages:
    print(msg.type, ":", msg.content)



--- Conversation End ---

🧠 Memory contents:
human : Hi FitBuddy, I like cardio and yoga workouts.
ai : That's great! Cardio and yoga complement each other well. Do you have any specific goals you’re working towards with these workouts?
human : Remind me what workouts I said I like?
ai : You mentioned that you like cardio and yoga workouts!
